In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from autofeat import AutoFeatRegressor, AutoFeatClassifier

df = pd.read_csv(r"~/mle_projects/mle-dvc/data/initial_data.csv")
df.columns

cat_features = [
    'paperless_billing',
    'payment_method',
    'internet_service',
    'online_security',
    'online_backup',
    'device_protection',
    'tech_support',
    'streaming_tv',
    'streaming_movies',
    'gender',
    'senior_citizen',
    'partner',
    'dependents',
    'multiple_lines',
]
num_features = ["monthly_charges", "total_charges"]
target = ["target"]
features = cat_features + num_features
split_column = "begin_date"
test_size = 0.2

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(
    df[features],
    df[target],
    test_size=test_size,
    shuffle=False,
)

transformations = ('1/', 'log', 'abs', 'sqrt')

afc = AutoFeatClassifier(
    categorical_cols=cat_features,
    transformations=transformations,
    feateng_steps=1,
    n_jobs=-1
)

X_train_features = afc.fit_transform(X_train, y_train)
X_test_features = afc.transform(X_test)

print(f"Train shape после автофичей: {X_train_features.shape}")
print(f"Test shape после автофичей: {X_test_features.shape}")

/home/mle-user/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/mle-user/.local/lib/python3.10/site-packages/autofeat/featsel.py:270: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  if np.max(np.abs(correlations[c].ravel()[:i])) < 0.9:


Train shape после автофичей: (5615, 33)
Test shape после автофичей: (1404, 33)


In [38]:
EXPERIMENT_NAME = "churn_prediction"
RUN_NAME = "new_fitch_V2" 
REGISTRY_MODEL_NAME = "churn_model_georgioparin"

In [29]:
import mlflow
import os
from dotenv import load_dotenv

load_dotenv("/home/mle-user/mle_projects/mle-mlflow/.env")
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ['AWS_ACCESS_KEY_ID'] = os.getenv('S3_ACCESS_KEY')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("S3_SECRET_KEY")

artifact_path = "afc"
experiment_id = mlflow.get_experiment_by_name("churn_prediction").experiment_id

with mlflow.start_run(run_name="new_fitch_V2", experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    afc_info = mlflow.sklearn.log_model(afc, artifact_path=artifact_path)

In [32]:
import mlflow

# experiment_id
experiment = mlflow.get_experiment_by_name("churn_prediction")
experiment_id = experiment.experiment_id

# найти последний run по имени (или просто самый последний по времени)
runs = mlflow.search_runs(experiment_ids=[experiment_id], order_by=["start_time DESC"])
target_run = runs[runs["tags.mlflow.runName"] == "new_fitch_V2"]

run_id = target_run.iloc[0]["run_id"]

# artifact_path — это то, что вы передавали в log_model
artifact_path = "afc"

# получить полный artifact_uri через клиента
client = mlflow.tracking.MlflowClient()
run_info = client.get_run(run_id)
artifact_uri = run_info.info.artifact_uri  # например: s3://bucket-name/experiment_id/run_id/artifacts

# bucket_name можно вытащить из artifact_uri
# формат обычно: s3://<bucket_name>/<experiment_id>/<run_id>/artifacts
bucket_name = artifact_uri.split("s3://")[1].split("/")[0]

print(f"bucket_name = {bucket_name}")
print(f"experiment_id = {experiment_id}")
print(f"run_id = {run_id}")
print(f"artifact_path = {artifact_path}")
print(f"artifact_uri (полный путь) = {artifact_uri}")

bucket_name = s3-student-mle-20260422-ac16c1f877-freetrack
experiment_id = 5
run_id = 6fdbb05586e748baa8aab38486c44aab
artifact_path = afc
artifact_uri (полный путь) = s3://s3-student-mle-20260422-ac16c1f877-freetrack/5/6fdbb05586e748baa8aab38486c44aab/artifacts


In [39]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score
import mlflow

y_train_s = y_train[target[0]] if hasattr(y_train, 'columns') else y_train
y_test_s = y_test[target[0]] if hasattr(y_test, 'columns') else y_test

model = CatBoostClassifier(iterations=150, verbose=0)
model.fit(X_train_features, y_train_s)

prediction = model.predict(X_test_features)
probas = model.predict_proba(X_test_features)[:, 1]

metrics = {
    "auc": roc_auc_score(y_test_s, probas),
    "precision": precision_score(y_test_s, prediction),
    "recall": recall_score(y_test_s, prediction)
}

print(metrics)

pip_requirements = "/home/mle-user/mle_projects/mle-mlflow/requirements.txt"
signature = mlflow.models.infer_signature(X_test_features, prediction)
input_example = X_test_features[:10]
metadata = {'model_type': 'monthly'}

EXPERIMENT_NAME = "churn_prediction"
RUN_NAME = "catboost_after_autofeat"
REGISTRY_MODEL_NAME = "churn_model_catboost"

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflow.log_metrics(metrics)

    model_info = mlflow.catboost.log_model(
        cb_model=model,
        artifact_path="models",
        pip_requirements=pip_requirements,
        signature=signature,
        input_example=input_example,
        metadata=metadata,
        registered_model_name=REGISTRY_MODEL_NAME,
        await_registration_for=60
    )

{'auc': 0.7417168797163673, 'precision': 0.551131221719457, 'recall': 0.90625}


/home/mle-user/.local/lib/python3.10/site-packages/_distutils_hack/__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
/home/mle-user/.local/lib/python3.10/site-packages/_distutils_hack/__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
Registered model 'churn_model_catboost' already exists. Creating a new version of this model...
2026/06/17 06:27:26 INFO mlflow.tracking._mod

In [41]:
model_registred_name = model_info.registered_model_version.name if hasattr(model_info, 'registered_model_version') else REGISTRY_MODEL_NAME
model_version_id = model_info.registered_model_version.version if hasattr(model_info, 'registered_model_version') else None
run_id = model_info.run_id

print(f"model_version_id = {model_version_id}")
print(f"model_registred_name = {model_registred_name}")
print(f"run_id = {run_id}")

model_version_id = None
model_registred_name = churn_model_catboost
run_id = 06b170fd637a4ed9ba05d5cd51a11268
